## Imports & setting up:

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM
from typing import Optional, Iterable, Dict, List
import os

import pandas as pd
import numpy as np
import warnings
import re
import matplotlib.pyplot as plt
import seaborn as sns
from cmcrameri import cm
import massbalancemachine as mbm
import logging
import torch.nn as nn
from skorch.helper import SliceDataset
from datetime import datetime
from skorch.callbacks import EarlyStopping, LRScheduler, Checkpoint
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset
import pickle 
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch 
from matplotlib.colors import LinearSegmentedColormap
import matplotlib as mpl
from matplotlib.ticker import FuncFormatter

from matplotlib.lines import Line2D
import scipy.stats as scipy_stats
from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"CROSS_REGION"
print(f"Base directory for data: {BASE_DIR}")

MONTHLY_COLS = [
    't2m',
    'tp',
    'slhf',
    'sshf',
    'ssrd',
    'fal',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope', 'svf']

feature_columns = MONTHLY_COLS + STATIC_COLS

# --- plot KDEs ---
COL_LABELS = {
    "t2m": "2m temperature",
    "tp": "Precipitation",
    "slhf": "Latent heat flux",
    "sshf": "Sensible heat flux",
    "ssrd": "Solar radiation",
    "fal": "Albedo",
    "str": "Thermal radiation",
    "ELEVATION_DIFFERENCE": "Elevation diff. (m)",
    "aspect": "Aspect (°)",
    "slope": "Slope (°)",
    "svf": "Sky-view factor",
}

# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc")
}

# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")


In [ ]:
# -----------------------------------------------------------------------
# Nature-compliant colorblind-safe palette
# Based on: https://research-figure-guide.nature.com/figures/building-and-exporting-figure-panels/
# -----------------------------------------------------------------------

NATURE_PALETTE = {
    "black": "#000000",
    "orange": "#e69f00",
    "sky_blue": "#56b4e9",
    "bluish_green": "#009e73",
    "yellow": "#f0e442",
    "blue": "#0072b2",
    "vermillion": "#d55e00",
    "reddish_purple": "#cc79a7",
}

# Convenient ordered list for sequential assignment
NATURE_COLORS = list(NATURE_PALETTE.values())

# Nature figure specs (for reference when setting figsize)
NATURE_SPECS = {
    "single_col_mm": 89,
    "double_col_mm": 183,
    "max_height_mm": 170,
    "font_min_pt": 5,
    "font_max_pt": 7,
    "font_family": "Arial",
    "dpi": 300,
}


def nature_figsize(cols=1, height_mm=80):
    """Return figsize in inches for Nature single or double column."""
    width_mm = NATURE_SPECS["single_col_mm"] if cols == 1 \
               else NATURE_SPECS["double_col_mm"]
    return (width_mm / 25.4, height_mm / 25.4)


def apply_nature_style(ax, fontsize=6, box=False):
    if box:
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.4)
    else:
        ax.spines[["top", "right"]].set_visible(False)
        ax.spines[["bottom", "left"]].set_linewidth(0.5)
    ax.tick_params(labelsize=fontsize, width=0.5, length=3)
    ax.xaxis.label.set_size(fontsize)
    ax.yaxis.label.set_size(fontsize)
    ax.title.set_size(fontsize + 1)
    ax.grid(color="#e0e0e0", linewidth=0.3, zorder=0)
    ax.set_axisbelow(True)


mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 6,
    "axes.linewidth": 0.5,
    "xtick.major.width": 0.5,
    "xtick.major.size": 3,
    "ytick.major.width": 0.5,
    "ytick.major.size": 3,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

nature_seq = LinearSegmentedColormap.from_list(
    "nature_seq",
    [
        NATURE_PALETTE["sky_blue"], NATURE_PALETTE["blue"],
        NATURE_PALETTE["black"]
    ],
)


## Load glacier grids:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (FR+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
dfs["11"].head(2)

### Switzerland:

In [ ]:
df_CH = load_stakes(cfg, "CH")
print(df_CH.GLACIER.unique())

# --- load aletsch monthly grid ---
glacier_name = 'aletsch'
rgi_id_aletsch = df_CH[df_CH.GLACIER == glacier_name].RGIId.values[0]

path_monthly_dfs = os.path.join(
    cfg.dataPath, 'RGI_v6/RGI_11_CentralEurope/monthly_parquets/')
path_parquet = os.path.join(path_monthly_dfs, rgi_id_aletsch)

yearly_files = sorted(
    glob.glob(os.path.join(path_parquet, f"{rgi_id_aletsch}_grid_*.parquet")))
print(f"Found {len(yearly_files)} yearly files for {glacier_name}")

df_aletsch = pd.concat([pd.read_parquet(f) for f in yearly_files],
                       ignore_index=True)
df_aletsch.dropna(subset=feature_columns, inplace=True)
print(f"Shape: {df_aletsch.shape}")

# df = df_aletsch[df_aletsch.YEAR == 2016]
# fig, ax = plt.subplots(figsize=nature_figsize(cols=1, height_mm=70))

# # Map the 'aspect' categories to Nature colors
# aspects = df["aspect"].dropna().unique()

# sns.scatterplot(
#     data=df,
#     x="POINT_LON",
#     y="POINT_LAT",
#     hue="aspect",
#     palette=nature_seq,
#     s=4,  # small markers suit Nature's dense figures
#     linewidth=0,  # no edge — cleaner at small sizes
#     alpha=0.8,
#     ax=ax,
# )

# apply_nature_style(ax, fontsize=6)  # strip top/right spines, grid, etc.

# ax.set_xlabel("Longitude")
# ax.set_ylabel("Latitude")
# ax.set_title(f"Aletsch – aspect (2016)", pad=4)

# # Tighten legend
# leg = ax.legend(
#     title="Aspect",
#     title_fontsize=6,
#     fontsize=5,
#     frameon=False,
#     markerscale=1.5,
#     handletextpad=0.4,
#     labelspacing=0.3,
# )

# plt.tight_layout()
# plt.show()

### Alaska:

In [ ]:
df_ALA = load_stakes(cfg, "ALA")
print(df_ALA.GLACIER.unique())

# --- load aletsch monthly grid ---
glacier_name = 'KAHILTNA'
rgi_id_kahil = df_ALA[df_ALA.GLACIER == glacier_name].RGIId.values[0]
print(rgi_id_kahil)
# rgi_id_kahil = "RGI60-01.00013"
# glacier_name = df_ALA[df_ALA.RGIId == rgi_id_kahil].GLACIER.values[0]
print(
    f"Loading monthly data for glacier {glacier_name} with RGI ID {rgi_id_kahil}"
)

path_monthly_dfs = os.path.join(cfg.dataPath,
                                'RGI_v6/RGI_01_Alaska/monthly_parquets')
path_parquet = os.path.join(path_monthly_dfs, rgi_id_kahil)

yearly_files = sorted(
    glob.glob(os.path.join(path_parquet, f"{rgi_id_kahil}_grid_*.parquet")))
print(f"Found {len(yearly_files)} yearly files for {glacier_name}")

df_gl_alaska = pd.concat([pd.read_parquet(f) for f in yearly_files],
                         ignore_index=True)
df_gl_alaska.dropna(subset=feature_columns, inplace=True)
print(f"Shape: {df_gl_alaska.shape}")

df = df_gl_alaska[df_gl_alaska.YEAR == 2016]
fig, ax = plt.subplots(figsize=nature_figsize(cols=1, height_mm=70))

# Map the 'aspect' categories to Nature colors
aspects = df["aspect"].dropna().unique()

sns.scatterplot(
    data=df,
    x="POINT_LON",
    y="POINT_LAT",
    hue="aspect",
    palette=nature_seq,
    s=4,  # small markers suit Nature's dense figures
    linewidth=0,  # no edge — cleaner at small sizes
    alpha=0.8,
    ax=ax,
)

apply_nature_style(ax, fontsize=6)  # strip top/right spines, grid, etc.

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"KAHILTNA – aspect (2016)", pad=4)

# Tighten legend
leg = ax.legend(
    title="Aspect",
    title_fontsize=6,
    fontsize=5,
    frameon=False,
    markerscale=1.5,
    handletextpad=0.4,
    labelspacing=0.3,
)

plt.tight_layout()
plt.show()

### Norway:

In [ ]:
df_NOR = load_stakes(cfg, "NOR")
print(df_NOR.GLACIER.unique())

# --- load aletsch monthly grid ---
glacier_name = 'Aalfotbreen'
rgi_id_aalfot = df_NOR[df_NOR.GLACIER == glacier_name].RGIId.values[0]

path_monthly_dfs = os.path.join(cfg.dataPath,
                                'RGI_v6/RGI_08_Scandinavia/monthly_parquets')
path_parquet = os.path.join(path_monthly_dfs, rgi_id_aalfot)

yearly_files = sorted(
    glob.glob(os.path.join(path_parquet, f"{rgi_id_aalfot}_grid_*.parquet")))
print(f"Found {len(yearly_files)} yearly files for {glacier_name}")

df_gl_nor = pd.concat([pd.read_parquet(f) for f in yearly_files],
                      ignore_index=True)
df_gl_nor.dropna(subset=feature_columns, inplace=True)
# print(f"Shape: {df_gl_nor.shape}")

# df = df_gl_nor[df_gl_nor.YEAR == 2016]
# fig, ax = plt.subplots(figsize=nature_figsize(cols=1, height_mm=70))

# # Map the 'aspect' categories to Nature colors
# aspects = df["aspect"].dropna().unique()

# sns.scatterplot(
#     data=df,
#     x="POINT_LON",
#     y="POINT_LAT",
#     hue="aspect",
#     palette=nature_seq,
#     s=4,  # small markers suit Nature's dense figures
#     linewidth=0,  # no edge — cleaner at small sizes
#     alpha=0.8,
#     ax=ax,
# )

# apply_nature_style(ax, fontsize=6)  # strip top/right spines, grid, etc.

# ax.set_xlabel("Longitude")
# ax.set_ylabel("Latitude")
# ax.set_title(f"{glacier_name} – aspect (2016)", pad=4)

# # Tighten legend
# leg = ax.legend(
#     title="Aspect",
#     title_fontsize=6,
#     fontsize=5,
#     frameon=False,
#     markerscale=1.5,
#     handletextpad=0.4,
#     labelspacing=0.3,
# )

# plt.tight_layout()
# plt.show()

### Iceland:

In [ ]:
# ── Iceland glacier selector ──────────────────────────────────────────────
# Pick one of the four options:
ISL_OPTIONS = {
    "RGI60-06.00340": {
        "name": "RGI60-06.00340",
        "rgi": "RGI60-06.00340"
    },
    "RGI60-06.00342": {
        "name": "RGI60-06.00342",
        "rgi": "RGI60-06.00342"
    },
    "RGI60-06.00328": {
        "name": "RGI60-06.00328",
        "rgi": "RGI60-06.00328"
    },
    "RGI60-06.00320": {
        "name": "Eyjafjallajokull",
        "rgi": "RGI60-06.00320"
    },
    "Oeldufellsjoekull": {
        "name": "Oeldufellsjoekull",
        "rgi": None
    },  # rgi looked up below
}

# ← CHANGE THIS to switch glacier
ISL_CHOICE = "RGI60-06.00320"

# ── Load chosen Iceland glacier ───────────────────────────────────────────
df_ISL = load_stakes(cfg, "ISL")

chosen = ISL_OPTIONS[ISL_CHOICE]
if chosen["rgi"] is not None:
    rgi_id_ISL = chosen["rgi"]
    #glacier_name_ISL = df_ISL[df_ISL.RGIId == rgi_id_ISL].GLACIER.values[0]
    glacier_name_ISL = chosen["name"]
else:
    # look up by name for Oeldufellsjoekull
    glacier_name_ISL = chosen["name"]
    rgi_id_ISL = df_ISL[df_ISL.GLACIER == glacier_name_ISL].RGIId.values[0]

print(f"Loading: {glacier_name_ISL} ({rgi_id_ISL})")

path_monthly_dfs = os.path.join(cfg.dataPath,
                                'RGI_v6/RGI_06_Iceland/monthly_parquets')
path_parquet = os.path.join(path_monthly_dfs, rgi_id_ISL)
yearly_files = sorted(
    glob.glob(os.path.join(path_parquet, f"{rgi_id_ISL}_grid_*.parquet")))
print(f"Found {len(yearly_files)} yearly files")

df_gl_ISL = pd.concat([pd.read_parquet(f) for f in yearly_files],
                      ignore_index=True)
df_gl_ISL.dropna(subset=feature_columns, inplace=True)
print(f"Shape: {df_gl_ISL.shape}")

## DEMS:

In [ ]:
# ── 0. Global elevation range across all three glaciers ──────────────────
datasets = {
    "aletsch":
    cfg.dataPath +
    f"/RGI_v6/RGI_11_CentralEurope/xr_masked_grids/{rgi_id_aletsch}.zarr",
    "kahiltna":
    cfg.dataPath +
    f"/RGI_v6/RGI_01_Alaska/xr_masked_grids/{rgi_id_kahil}.zarr",
    "aalfot":
    cfg.dataPath +
    f"/RGI_v6/RGI_08_Scandinavia/xr_masked_grids/{rgi_id_aalfot}.zarr",
    "isl":
    cfg.dataPath + f"/RGI_v6/RGI_06_Iceland/xr_masked_grids/{rgi_id_ISL}.zarr",
}

vmin, vmax = np.inf, -np.inf
for path in datasets.values():
    elev = xr.open_dataset(path).masked_elev
    # match robust=True: use 2nd–98th percentile
    vmin = min(vmin, float(elev.quantile(0.02)))
    vmax = max(vmax, float(elev.quantile(0.98)))

print(f"Global elevation range: {vmin:.0f} – {vmax:.0f} m")

elev_cmap = plt.cm.terrain
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

In [ ]:
# ── 1. Reusable plot function ─────────────────────────────────────────────
def plot_dem(path, title, fname):
    ds = xr.open_dataset(path)
    fig, ax = plt.subplots(figsize=nature_figsize(cols=1, height_mm=80))

    ds.masked_elev.plot(
        ax=ax,
        cmap=elev_cmap,
        vmin=vmin,
        vmax=vmax,
        add_colorbar=False,
    )

    apply_nature_style(ax, fontsize=6)

    # override apply_nature_style — restore full box
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.4)

    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(length=0)
    ax.set_title(title, pad=4, fontsize=7)

    plt.tight_layout()
    plt.savefig(f"figures/paperTF/{fname}.pdf", bbox_inches="tight")
    # plt.savefig(f"figures/paperTF/{fname}.png", dpi=300, bbox_inches="tight")
    plt.show()


plot_dem(datasets["aletsch"], "", "aletsch_DEM")
plot_dem(datasets["kahiltna"], "", "kahiltna_DEM")
plot_dem(datasets["aalfot"], "", "aalfot_DEM")
plot_dem(datasets["isl"], "", "isl_DEM")

In [ ]:
# # ── 3-panel DEM figure (same height as KDE plot) ─────────────────────────
# dem_order = [
#     ("aalfot", "Ålfotbreen (NOR)"),
#     ("aletsch", "Grosser Aletschgletscher (CH)"),
#     ("kahiltna", "Kahiltna Glacier (US)"),
#     ("isl", "Oeldufellsjoekull (ISL)"),
# ]

# fig_dem, axes_dem = plt.subplots(
#     3,
#     1,
#     figsize=(60 / 25.4, 200 / 25.4),  # narrow, same height as KDE
# )

# for ax, (key, title) in zip(axes_dem, dem_order):
#     ds = xr.open_dataset(datasets[key])
#     ds.masked_elev.plot(
#         ax=ax,
#         cmap=elev_cmap,
#         vmin=vmin,
#         vmax=vmax,
#         add_colorbar=False,
#     )

#     apply_nature_style(ax, fontsize=6, box=True)
#     ax.set_xticklabels([])
#     ax.set_yticklabels([])
#     ax.set_xlabel("")
#     ax.set_ylabel("")
#     ax.tick_params(length=0)
#     ax.set_title(title, pad=3, fontsize=6, loc="left")

# plt.tight_layout(pad=0.5, h_pad=1.0)
# plt.savefig("figures/paperTF/all_DEMs.pdf", bbox_inches="tight")
# # plt.savefig("figures/paperTF/all_DEMs.png", dpi=300, bbox_inches="tight")
# plt.show()

In [ ]:
import matplotlib.colorbar as mcolorbar
# ── 2. Standalone colorbar ────────────────────────────────────────────────
fig_cb, ax_cb = plt.subplots(figsize=(2, 0.2))  # wide & short

cb = mcolorbar.ColorbarBase(
    ax_cb,
    cmap=elev_cmap,
    norm=norm,
    orientation="horizontal",
)
cb.set_label("Elevation (m a.s.l.)", fontsize=6)
cb.ax.tick_params(labelsize=5, width=0.5, length=3)
cb.outline.set_linewidth(0.5)

plt.tight_layout()
plt.savefig("figures/paperTF/shared_colorbar.pdf", bbox_inches="tight")
plt.savefig("figures/paperTF/shared_colorbar.png",
            dpi=300,
            bbox_inches="tight")
plt.show()

## KDE plot:

In [ ]:
GLACIERS_TO_PLOT = {
    "Aletsch (CH)": {
        "df":    df_aletsch,
        "color": NATURE_PALETTE["blue"],
        "rgi":   "RGI60-11.01450",
        "extent": (5.8, 11, 45.5, 48.0),
        "gdf":   gpd.read_file(os.path.join(cfg.dataPath,
                     "RGI_v6/RGI_11_CentralEurope/11_rgi60_CentralEurope.shp")),
    },
    "Ålfotbreen (NOR)": {
        "df":    df_gl_nor,
        "color": NATURE_PALETTE["vermillion"],
        "rgi":   "RGI60-08.02992",
        "extent": (4.0, 24.0, 58.0, 71.0),
        "gdf":   gpd.read_file(os.path.join(cfg.dataPath,
                     "RGI_v6/RGI_08_Scandinavia/08_rgi60_Scandinavia.shp")),
    },
    "Kahiltna (ALA)": {
        "df":    df_gl_alaska,
        "color": NATURE_PALETTE["bluish_green"],
        "rgi":   "RGI60-01.00013",
        "extent": (-170, -142, 55.0, 68.0),
        "gdf":   gpd.read_file(os.path.join(cfg.dataPath,
                     "RGI_v6/RGI_01_Alaska/01_rgi60_Alaska.shp")),
    },
    f"{glacier_name_ISL} (ISL)": {           # ← label uses chosen name
        "df":    df_gl_ISL,
        "color": NATURE_PALETTE["reddish_purple"],   # 4th Nature color
        "rgi":   rgi_id_ISL,
        "extent": (-25.0, -13.0, 63.0, 67.0),       # Iceland bounding box
        "gdf":   gpd.read_file(os.path.join(cfg.dataPath,
                     "RGI_v6/RGI_06_Iceland/06_rgi60_Iceland.shp")),
    },
}

COLORS = {label: cfg_gl["color"] for label, cfg_gl in GLACIERS_TO_PLOT.items()}

# selected features for the KDE panel
SELECTED_COLS = [
    "t2m", "tp", "ssrd", "sshf", "ELEVATION_DIFFERENCE", "aspect", "slope",
    "svf"
]
COL_LABELS = {
    "t2m": "2m temperature (°C)",
    "tp": "Precipitation (m w.e.)",
    "ELEVATION_DIFFERENCE": "Elevation diff. (m)",
    "aspect": "Aspect (rad)",
    "ssrd": "Solar radiation (J/m²)",
    "svf": "Sky-view factor",
    "slope": "Slope (rad)",
    "sshf": "Sensible heat flux (J/m²)",
}


In [ ]:
# font fallback — use DejaVu Sans if Arial not available
try:
    mpl.font_manager.findfont("Arial", fallback_to_default=False)
    mpl.rcParams["font.family"] = "Arial"
except:
    mpl.rcParams["font.family"] = "DejaVu Sans"  # closest available


def format_axis_ticks(ax, label_size=8):
    """Format tick labels to avoid huge 1e6/1e7 offset labels."""
    # check if scientific notation offset is being used
    ax.xaxis.get_major_formatter().set_useOffset(False)
    try:
        # scale large numbers to readable units
        xmax = abs(ax.get_xlim()[1])
        if xmax > 1e6:
            scale = 1e6
            ax.xaxis.set_major_formatter(
                FuncFormatter(lambda x, _: f"{x/scale:.1f}"))
            ax.set_xlabel(f"(×10⁶)", label_size=label_size, labelpad=1)
        elif xmax > 1e4:
            scale = 1e3
            ax.xaxis.set_major_formatter(
                FuncFormatter(lambda x, _: f"{x/scale:.0f}"))
            ax.set_xlabel(f"(×10³)", label_size=label_size, labelpad=1)
    except Exception:
        pass

In [ ]:
ncols = 4
nrows = int(np.ceil(len(SELECTED_COLS) / ncols))

w, h = nature_figsize(cols=1, height_mm=100)
fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(w * 2, h),  # 178mm wide, 100mm tall
    squeeze=False,
)

for idx, col in enumerate(SELECTED_COLS):
    ax = axes[idx // ncols][idx % ncols]

    all_vals = pd.concat(
        [cfg_gl["df"][col].dropna() for cfg_gl in GLACIERS_TO_PLOT.values()])
    xmin = float(all_vals.min())
    xmax = float(all_vals.max())
    x_grid = np.linspace(xmin, xmax, 500)

    for label, cfg_gl in GLACIERS_TO_PLOT.items():
        vals = cfg_gl["df"][col].dropna().values
        if len(vals) < 10:
            continue
        kde = scipy_stats.gaussian_kde(vals, bw_method=0.3)
        y = kde(x_grid)
        y = y / y.max()
        ax.plot(x_grid, y, color=cfg_gl["color"], linewidth=0.8, label=label)
        ax.fill_between(x_grid, y, alpha=0.08, color=cfg_gl["color"])

    ax.set_ylim(0, 1.15)
    ax.set_xlabel("")
    ax.tick_params(labelsize=8, width=0.4, length=2, direction="in")
    ax.spines[["top", "right", "left", "bottom"]].set_visible(True)
    for spine in ax.spines.values():
        spine.set_linewidth(0.4)
    ax.grid(axis="x", color="#e0e0e0", linewidth=0.3)
    ax.set_axisbelow(True)

    format_axis_ticks(ax, label_size=6)

for idx in range(len(SELECTED_COLS), nrows * ncols):
    axes[idx // ncols][idx % ncols].axis("off")

handles = [
    plt.Line2D([0], [0], color=COLORS[l], linewidth=1.0, label=l)
    for l in GLACIERS_TO_PLOT
]
plt.tight_layout(h_pad=3.0)
plt.savefig("figures/paperTF/section1_kde_nature.pdf", bbox_inches="tight")
plt.savefig("figures/paperTF/section1_kde_nature.png",
            dpi=300,
            bbox_inches="tight")
plt.show()

In [ ]:
# ── Standalone legend ─────────────────────────────────────────────────────
fig_leg, ax_leg = plt.subplots(figsize=(130 / 25.4,
                                        10 / 25.4))  # wide, very short
ax_leg.axis("off")

handles = [
    plt.Line2D([0], [0], color=COLORS[l], linewidth=1.0, label=l)
    for l in GLACIERS_TO_PLOT
]

ax_leg.legend(
    handles=handles,
    loc="center",
    ncol=len(GLACIERS_TO_PLOT),
    frameon=False,
    fontsize=14,
    handlelength=1.5,
    columnspacing=1.0,
)

plt.tight_layout(pad=0.1)
plt.savefig("figures/paperTF/kde_panels/kde_legend.pdf", bbox_inches="tight")
plt.savefig("figures/paperTF/kde_panels/kde_legend.png",
            dpi=300,
            bbox_inches="tight")
plt.close()
print("Saved standalone legend.")

In [ ]:
os.makedirs("figures/paperTF/kde_panels", exist_ok=True)


def _draw_kde_panel(ax, col, GLACIERS_TO_PLOT):
    """Draw a single KDE panel onto ax. Returns the x_grid for reuse."""
    all_vals = pd.concat(
        [cfg_gl["df"][col].dropna() for cfg_gl in GLACIERS_TO_PLOT.values()])
    x_grid = np.linspace(float(all_vals.min()), float(all_vals.max()), 500)

    for label, cfg_gl in GLACIERS_TO_PLOT.items():
        vals = cfg_gl["df"][col].dropna().values
        if len(vals) < 10:
            continue
        kde = scipy_stats.gaussian_kde(vals, bw_method=0.3)
        y = kde(x_grid)
        y = y / y.max()
        ax.plot(x_grid, y, color=cfg_gl["color"], linewidth=0.8, label=label)
        ax.fill_between(x_grid, y, alpha=0.08, color=cfg_gl["color"])

    ax.set_ylim(0, 1.15)
    ax.set_xlabel("")
    ax.tick_params(labelsize=10, width=0.4, length=2)
    ax.spines[["top", "right", "left", "bottom"]].set_visible(True)
    for spine in ax.spines.values():
        spine.set_linewidth(0.4)
    ax.grid(axis="x", color="#e0e0e0", linewidth=0.3)
    ax.set_axisbelow(True)
    format_axis_ticks(ax, label_size=10)


# ── 2. Individual panels ──────────────────────────────────────────────────
for col in SELECTED_COLS:
    fig_s, ax_s = plt.subplots(figsize=(95 / 25.4,
                                        80 / 25.4))  # full single column width
    _draw_kde_panel(ax_s, col, GLACIERS_TO_PLOT)
    plt.tight_layout(pad=0.4)
    fname = col.lower().replace(" ", "_")
    plt.savefig(f"figures/paperTF/kde_panels/kde_{fname}.pdf",
                bbox_inches="tight")
    plt.savefig(f"figures/paperTF/kde_panels/kde_{fname}.png",
                dpi=300,
                bbox_inches="tight")
    plt.close()
print(
    f"Saved {len(SELECTED_COLS)} individual panels to figures/paperTF/kde_panels/"
)

## Compute sinkhorn distances:

In [ ]:
# CLIMATE_COLS = [
#     't2m', 'tp', 'slhf', 'sshf', 'ssrd', 'fal', 'str', 'ELEVATION_DIFFERENCE'
# ]
# TOPO_COLS = ['aspect', 'slope', 'svf']

# glacier_labels = list(GLACIERS_TO_PLOT.keys())
# glacier_shifts = {}

# for src_label in tqdm(glacier_labels, desc="glacier pairs"):
#     for tgt_label in glacier_labels:
#         if src_label == tgt_label:
#             continue
#         shift = compute_domain_shift(
#             df_src=GLACIERS_TO_PLOT[src_label]["df"],
#             df_tgt=GLACIERS_TO_PLOT[tgt_label]["df"],
#             monthly_cols=CLIMATE_COLS,
#             static_cols=TOPO_COLS,
#             # No scalers, no blurs passed → fit locally per pair
#             seed=cfg.seed,
#         )
#         glacier_shifts[(src_label, tgt_label)] = shift
#         print(
#             f"{src_label} → {tgt_label}: {shift['D_sinkhorn_joint_true']:.4f}")

# dist_matrix = pd.DataFrame(index=glacier_labels,
#                            columns=glacier_labels,
#                            dtype=float)
# for (src_label, tgt_label), shift in glacier_shifts.items():
#     dist_matrix.loc[src_label, tgt_label] = shift.get("D_sinkhorn_joint_true",
#                                                       float("nan"))

# print("\nPairwise Sinkhorn distances between glaciers:")
# print(dist_matrix.round(3).to_string())

In [ ]:
# # Simple global scaler: just stack all glacier dfs and fit directly
# df_all = pd.concat([cfg_gl["df"] for cfg_gl in GLACIERS_TO_PLOT.values()],
#                    ignore_index=True)

# scaler_m_gl = StandardScaler().fit(
#     df_all[CLIMATE_COLS].to_numpy(dtype=np.float64))
# scaler_s_gl = StandardScaler().fit(
#     df_all[TOPO_COLS].to_numpy(dtype=np.float64))

# print("Climate scaler means:", scaler_m_gl.mean_.round(4))
# print("Climate scaler stds: ", scaler_m_gl.scale_.round(4))
# print("Topo scaler means:   ", scaler_s_gl.mean_.round(4))
# print("Topo scaler stds:    ", scaler_s_gl.scale_.round(4))

# # Then rerun shifts with these scalers
# glacier_labels = list(GLACIERS_TO_PLOT.keys())
# glacier_shifts_simple = {}

# for src_label in tqdm(glacier_labels, desc="glacier pairs"):
#     for tgt_label in glacier_labels:
#         if src_label == tgt_label:
#             continue
#         shift = compute_domain_shift(
#             df_src=GLACIERS_TO_PLOT[src_label]["df"],
#             df_tgt=GLACIERS_TO_PLOT[tgt_label]["df"],
#             monthly_cols=CLIMATE_COLS,
#             static_cols=TOPO_COLS,
#             scaler_m=scaler_m_gl,
#             scaler_s=scaler_s_gl,
#             seed=cfg.seed,
#         )
#         glacier_shifts_simple[(src_label, tgt_label)] = shift
#         print(
#             f"{src_label} → {tgt_label}: {shift['D_sinkhorn_joint_true']:.4f}")

# dist_matrix_simple = pd.DataFrame(index=glacier_labels,
#                                   columns=glacier_labels,
#                                   dtype=float)
# for (src_label, tgt_label), shift in glacier_shifts_simple.items():
#     dist_matrix_simple.loc[src_label,
#                            tgt_label] = shift.get("D_sinkhorn_joint_true",
#                                                   float("nan"))

# print("\nPairwise Sinkhorn distances (simple global scaler):")
# print(dist_matrix_simple.round(3).to_string())

In [ ]:
# Simple global scaler: just stack all glacier dfs and fit directly
df_all = pd.concat([cfg_gl["df"] for cfg_gl in GLACIERS_TO_PLOT.values()],
                   ignore_index=True)

scaler_m_gl = StandardScaler().fit(
    df_all[CLIMATE_COLS].to_numpy(dtype=np.float64))
scaler_s_gl = StandardScaler().fit(
    df_all[TOPO_COLS].to_numpy(dtype=np.float64))

print("Climate scaler means:", scaler_m_gl.mean_.round(4))
print("Climate scaler stds: ", scaler_m_gl.scale_.round(4))
print("Topo scaler means:   ", scaler_s_gl.mean_.round(4))
print("Topo scaler stds:    ", scaler_s_gl.scale_.round(4))

glacier_dfs = {
    label: cfg_gl["df"]
    for label, cfg_gl in GLACIERS_TO_PLOT.items()
}

blur_m_gl, blur_s_gl, blur_joint_gl = estimate_global_bandwidths_from_dfs(
    dfs=glacier_dfs,
    monthly_cols=CLIMATE_COLS,
    static_cols=TOPO_COLS,
    scaler_m=scaler_m_gl,
    scaler_s=scaler_s_gl,
    seed=cfg.seed,
)
bandwidths_m_gl = [blur_m_gl * 0.5, blur_m_gl, blur_m_gl * 2.0]
bandwidths_s_gl = [blur_s_gl * 0.5, blur_s_gl, blur_s_gl * 2.0]
print(f"Glacier blurs: blur_m={blur_m_gl:.4f}, "
      f"blur_s={blur_s_gl:.4f}, blur_joint={blur_joint_gl:.4f}")

# Then rerun shifts with these scalers
glacier_labels = list(GLACIERS_TO_PLOT.keys())
glacier_shifts_simple = {}

for src_label in tqdm(glacier_labels, desc="glacier pairs"):
    for tgt_label in glacier_labels:
        if src_label == tgt_label:
            continue
        shift = compute_domain_shift(
            df_src=GLACIERS_TO_PLOT[src_label]["df"],
            df_tgt=GLACIERS_TO_PLOT[tgt_label]["df"],
            monthly_cols=CLIMATE_COLS,
            static_cols=TOPO_COLS,
            scaler_m=scaler_m_gl,
            scaler_s=scaler_s_gl,
            seed=cfg.seed,
            blur_m=blur_m_gl,
            blur_s=blur_s_gl,
            blur_joint=blur_joint_gl,
            bandwidths_m=bandwidths_m_gl,
            bandwidths_s=bandwidths_s_gl,
        )
        glacier_shifts_simple[(src_label, tgt_label)] = shift
        print(
            f"{src_label} → {tgt_label}: {shift['D_sinkhorn_joint_true']:.4f}")

dist_matrix_simple = pd.DataFrame(index=glacier_labels,
                                  columns=glacier_labels,
                                  dtype=float)
for (src_label, tgt_label), shift in glacier_shifts_simple.items():
    dist_matrix_simple.loc[src_label,
                           tgt_label] = shift.get("D_sinkhorn_joint_true",
                                                  float("nan"))

print("\nPairwise Sinkhorn distances (simple global scaler):")
print(dist_matrix_simple.round(3).to_string())

In [ ]:
CLIMATE_COLS = [
    't2m', 'tp', 'slhf', 'sshf', 'ssrd', 'fal', 'str', 'ELEVATION_DIFFERENCE'
]
TOPO_COLS = ['aspect', 'slope', 'svf']

glacier_dfs = {
    label: cfg_gl["df"]
    for label, cfg_gl in GLACIERS_TO_PLOT.items()
}

scaler_m_gl, scaler_s_gl, scaler_all_gl = build_global_scalers_from_dfs(
    dfs=glacier_dfs,
    monthly_cols=CLIMATE_COLS,
    static_cols=TOPO_COLS,
)

blur_m_gl, blur_s_gl, blur_joint_gl = estimate_global_bandwidths_from_dfs(
    dfs=glacier_dfs,
    monthly_cols=CLIMATE_COLS,
    static_cols=TOPO_COLS,
    scaler_m=scaler_m_gl,
    scaler_s=scaler_s_gl,
    seed=cfg.seed,
)
bandwidths_m_gl = [blur_m_gl * 0.5, blur_m_gl, blur_m_gl * 2.0]
bandwidths_s_gl = [blur_s_gl * 0.5, blur_s_gl, blur_s_gl * 2.0]
print(f"Glacier blurs: blur_m={blur_m_gl:.4f}, "
      f"blur_s={blur_s_gl:.4f}, blur_joint={blur_joint_gl:.4f}")

# --- Compute pairwise Sinkhorn distances between all glacier pairs ---
glacier_labels = list(GLACIERS_TO_PLOT.keys())
glacier_shifts = {}

for src_label in tqdm(glacier_labels, desc="glacier pairs"):
    for tgt_label in glacier_labels:
        if src_label == tgt_label:
            continue
        shift = compute_domain_shift(
            df_src=GLACIERS_TO_PLOT[src_label]["df"],
            df_tgt=GLACIERS_TO_PLOT[tgt_label]["df"],
            monthly_cols=CLIMATE_COLS,
            static_cols=TOPO_COLS,
            scaler_m=scaler_m_gl,
            scaler_s=scaler_s_gl,
            blur_m=blur_m_gl,
            blur_s=blur_s_gl,
            blur_joint=blur_joint_gl,
            bandwidths_m=bandwidths_m_gl,
            bandwidths_s=bandwidths_s_gl,
            seed=cfg.seed,
        )
        glacier_shifts[(src_label, tgt_label)] = shift
        print(f"{src_label} → {tgt_label}: {shift['D_sinkhorn_joint']:.4f}")

# --- Symmetric distance matrix ---
dist_matrix = pd.DataFrame(index=glacier_labels,
                           columns=glacier_labels,
                           dtype=float)
for (src_label, tgt_label), shift in glacier_shifts.items():
    dist_matrix.loc[src_label, tgt_label] = shift.get("D_sinkhorn_joint",
                                                      float("nan"))

print("\nPairwise Sinkhorn distances between glaciers:")
print(dist_matrix.round(3).to_string())


In [ ]:
# --- Glacier pairwise Sinkhorn heatmap ---
glacier_labels = list(GLACIERS_TO_PLOT.keys())
glacier_labels_rev = list(reversed(glacier_labels))

gl_matrix = pd.DataFrame(index=glacier_labels,
                         columns=glacier_labels_rev,
                         dtype=float)
for src_label in glacier_labels:
    for tgt_label in glacier_labels_rev:
        if src_label == tgt_label:
            continue
        val = glacier_shifts.get((src_label, tgt_label), {})
        gl_matrix.loc[src_label, tgt_label] = val.get("D_sinkhorn_joint_true",
                                                      float("nan"))

gl_mask = pd.DataFrame(False, index=glacier_labels, columns=glacier_labels_rev)
for i, r in enumerate(glacier_labels):
    for j, c in enumerate(glacier_labels_rev):
        if j >= len(glacier_labels) - i:  # diagonal and right of it
            gl_mask.loc[r, c] = True

fig, ax = plt.subplots(figsize=nature_figsize(cols=2, height_mm=70))
cmap = sns.dark_palette("#69d", reverse=True, as_cmap=True)

sns.heatmap(
    gl_matrix.astype(float),
    mask=gl_mask,
    ax=ax,
    cmap=cmap,
    annot=True,
    fmt=".2f",
    linewidths=0.4,
    linecolor="#dddddd",
    annot_kws={"fontsize": NATURE_SPECS["font_min_pt"]},
    cbar_kws={
        "label": "Sinkhorn joint distance",
        "shrink": 0.8
    },
)

ax.set_xlabel("Glacier", fontsize=NATURE_SPECS["font_min_pt"])
ax.set_ylabel("Glacier", fontsize=NATURE_SPECS["font_min_pt"])
ax.tick_params(labelsize=NATURE_SPECS["font_min_pt"], length=0)
ax.set_title(
    "Pairwise topoclimatic distance between glaciers (Sinkhorn joint)",
    fontsize=NATURE_SPECS["font_max_pt"],
)

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=NATURE_SPECS["font_min_pt"])
cbar.set_label("Sinkhorn joint distance", fontsize=NATURE_SPECS["font_min_pt"])

plt.tight_layout()
plt.savefig(
    "figures/paperTF/section1_glacier_shift_heatmap.png",
    dpi=NATURE_SPECS["dpi"],
    bbox_inches="tight",
)
plt.show()

In [ ]:
# --- Glacier pairwise Sinkhorn heatmap ---
glacier_labels = list(GLACIERS_TO_PLOT.keys())
glacier_labels_rev = list(reversed(glacier_labels))

gl_matrix = pd.DataFrame(index=glacier_labels,
                         columns=glacier_labels_rev,
                         dtype=float)
for src_label in glacier_labels:
    for tgt_label in glacier_labels_rev:
        if src_label == tgt_label:
            continue
        val = glacier_shifts.get((src_label, tgt_label), {})
        gl_matrix.loc[src_label, tgt_label] = val.get("D_sinkhorn_topo",
                                                      float("nan"))

gl_mask = pd.DataFrame(False, index=glacier_labels, columns=glacier_labels_rev)
for i, r in enumerate(glacier_labels):
    for j, c in enumerate(glacier_labels_rev):
        if j >= len(glacier_labels) - i:  # diagonal and right of it
            gl_mask.loc[r, c] = True

fig, ax = plt.subplots(figsize=nature_figsize(cols=2, height_mm=70))
cmap = sns.dark_palette("#69d", reverse=True, as_cmap=True)

sns.heatmap(
    gl_matrix.astype(float),
    mask=gl_mask,
    ax=ax,
    cmap=cmap,
    annot=True,
    fmt=".2f",
    linewidths=0.4,
    linecolor="#dddddd",
    annot_kws={"fontsize": NATURE_SPECS["font_min_pt"]},
    cbar_kws={
        "label": "Sinkhorn joint distance",
        "shrink": 0.8
    },
)

ax.set_xlabel("Glacier", fontsize=NATURE_SPECS["font_min_pt"])
ax.set_ylabel("Glacier", fontsize=NATURE_SPECS["font_min_pt"])
ax.tick_params(labelsize=NATURE_SPECS["font_min_pt"], length=0)
ax.set_title(
    "Pairwise topoclimatic distance between glaciers (Sinkhorn topo)",
    fontsize=NATURE_SPECS["font_max_pt"],
)

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=NATURE_SPECS["font_min_pt"])
cbar.set_label("Sinkhorn topo distance", fontsize=NATURE_SPECS["font_min_pt"])

plt.tight_layout()
plt.savefig(
    "figures/paperTF/section1_glacier_shift_heatmap.png",
    dpi=NATURE_SPECS["dpi"],
    bbox_inches="tight",
)
plt.show()

In [ ]:
# --- Glacier pairwise Sinkhorn heatmap ---
glacier_labels = list(GLACIERS_TO_PLOT.keys())
glacier_labels_rev = list(reversed(glacier_labels))

gl_matrix = pd.DataFrame(index=glacier_labels,
                         columns=glacier_labels_rev,
                         dtype=float)
for src_label in glacier_labels:
    for tgt_label in glacier_labels_rev:
        if src_label == tgt_label:
            continue
        val = glacier_shifts.get((src_label, tgt_label), {})
        gl_matrix.loc[src_label, tgt_label] = val.get("D_sinkhorn_climate",
                                                      float("nan"))

gl_mask = pd.DataFrame(False, index=glacier_labels, columns=glacier_labels_rev)
for i, r in enumerate(glacier_labels):
    for j, c in enumerate(glacier_labels_rev):
        if j >= len(glacier_labels) - i:  # diagonal and right of it
            gl_mask.loc[r, c] = True

fig, ax = plt.subplots(figsize=nature_figsize(cols=2, height_mm=70))
cmap = sns.dark_palette("#69d", reverse=True, as_cmap=True)

sns.heatmap(
    gl_matrix.astype(float),
    mask=gl_mask,
    ax=ax,
    cmap=cmap,
    annot=True,
    fmt=".2f",
    linewidths=0.4,
    linecolor="#dddddd",
    annot_kws={"fontsize": NATURE_SPECS["font_min_pt"]},
    cbar_kws={
        "label": "Sinkhorn joint distance",
        "shrink": 0.8
    },
)

ax.set_xlabel("Glacier", fontsize=NATURE_SPECS["font_min_pt"])
ax.set_ylabel("Glacier", fontsize=NATURE_SPECS["font_min_pt"])
ax.tick_params(labelsize=NATURE_SPECS["font_min_pt"], length=0)
ax.set_title(
    "Pairwise climatic distance between glaciers (Sinkhorn climate)",
    fontsize=NATURE_SPECS["font_max_pt"],
)

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=NATURE_SPECS["font_min_pt"])
cbar.set_label("Sinkhorn climate distance",
               fontsize=NATURE_SPECS["font_min_pt"])

plt.tight_layout()
plt.savefig(
    "figures/paperTF/section1_glacier_shift_heatmap.png",
    dpi=NATURE_SPECS["dpi"],
    bbox_inches="tight",
)
plt.show()

## Maps:

In [ ]:
def plot_glacier_outline_map(
    glacier_outline_rgi,
    highlight_rgi_id=None,
    highlight_color="#2166ac",
    *,
    title="",
    figsize=nature_figsize(cols=1, height_mm=55),
    extent=(5.8, 15, 44, 48),
    land_facecolor="#f0f0f0",
    land_alpha=1.0,
    outline_color="#555555",
    outline_alpha=0.6,
    add_features=True,
    add_gridlines=False,
    show=True,
):
    lonW, lonE, latS, latN = extent
    projPC = ccrs.PlateCarree()

    fig = plt.figure(figsize=figsize)
    ax = plt.axes(projection=projPC)
    ax.set_extent([lonW, lonE, latS, latN], crs=ccrs.Geodetic())

    if add_features:
        ax.add_feature(cfeature.OCEAN, facecolor="#daeeff", zorder=0)
        ax.add_feature(cfeature.LAND,
                       facecolor=land_facecolor,
                       alpha=land_alpha,
                       zorder=0)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.3, zorder=1)
        ax.add_feature(cfeature.BORDERS,
                       linestyle="-",
                       linewidth=0.2,
                       zorder=1)
        ax.add_feature(cfeature.LAKES, linewidth=0.2, zorder=1)
        ax.add_feature(cfeature.RIVERS, linewidth=0.2, zorder=1)

    if add_gridlines:
        gl = ax.gridlines(draw_labels=True,
                          linewidth=0.2,
                          color="gray",
                          alpha=0.4,
                          linestyle="--")
        gl.top_labels = gl.right_labels = False
        gl.xlabel_style = {"size": 5}
        gl.ylabel_style = {"size": 5}

    # --- all glacier outlines clipped to extent ---
    gdf_clipped = glacier_outline_rgi.cx[lonW:lonE, latS:latN]
    if not gdf_clipped.empty:
        gdf_clipped.plot(ax=ax,
                         transform=projPC,
                         color=outline_color,
                         alpha=outline_alpha,
                         linewidth=0.2)

    # --- star marker on target glacier ---
    if highlight_rgi_id is not None:
        gdf_target = glacier_outline_rgi[glacier_outline_rgi["RGIId"] ==
                                         highlight_rgi_id]
        if not gdf_target.empty:
            centroid = gdf_target.geometry.centroid.iloc[0]
            ax.plot(
                centroid.x,
                centroid.y,
                marker="*",
                markersize=20,
                color=highlight_color,
                markeredgecolor="white",
                markeredgewidth=0.3,
                transform=projPC,
                zorder=4,
            )
        else:
            print(f"Warning: RGIId {highlight_rgi_id} not found in shapefile.")

    if title:
        ax.set_title(title,
                     fontsize=14,
                     fontweight="bold",
                     loc="left",
                     pad=2,
                     color=highlight_color)

    plt.tight_layout(pad=0.2)
    if show:
        plt.show()

    return fig, ax

In [ ]:
for label, cfg_gl in GLACIERS_TO_PLOT.items():
    fig, ax = plot_glacier_outline_map(
        glacier_outline_rgi=cfg_gl["gdf"],
        highlight_rgi_id=cfg_gl["rgi"],
        highlight_color=cfg_gl["color"],
        title=label,
        extent=cfg_gl["extent"],
        figsize=(6, 5),
        show=False,
    )
    fname = label.replace(" ", "_").replace("(",
                                            "").replace(")",
                                                        "").replace("/", "_")
    plt.savefig(f"figures/paperTF/map_{fname}.pdf", bbox_inches="tight")
    plt.savefig(f"figures/paperTF/map_{fname}.png",
                dpi=300,
                bbox_inches="tight")
    plt.close()
    print(f"Saved map_{fname}")

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig = plt.figure(figsize=nature_figsize(cols=2, height_mm=80))
ax = plt.axes(projection=ccrs.Robinson())
ax.set_global()

# ── base map ──────────────────────────────────────────────────────────────
ax.add_feature(cfeature.OCEAN, facecolor="#daeeff", zorder=0)
ax.add_feature(cfeature.LAND, facecolor="#f0f0f0", zorder=0)
ax.add_feature(cfeature.COASTLINE, linewidth=0.3, zorder=1)
ax.add_feature(cfeature.BORDERS, linewidth=0.2, zorder=1)

# ── glacier centroids ─────────────────────────────────────────────────────
for label, cfg_gl in GLACIERS_TO_PLOT.items():
    gdf_target = cfg_gl["gdf"][cfg_gl["gdf"]["RGIId"] == cfg_gl["rgi"]]
    if gdf_target.empty:
        print(f"Warning: {cfg_gl['rgi']} not found, skipping.")
        continue
    centroid = gdf_target.geometry.centroid.iloc[0]
    ax.plot(
        centroid.x,
        centroid.y,
        marker="*",
        markersize=10,
        color=cfg_gl["color"],
        markeredgecolor="white",
        markeredgewidth=0.3,
        transform=ccrs.PlateCarree(),
        zorder=4,
        label=label,
    )

# ── legend ────────────────────────────────────────────────────────────────
ax.legend(
    loc="lower left",
    fontsize=5,
    frameon=False,
    markerscale=1.0,
    handletextpad=0.3,
)

apply_nature_style(ax, fontsize=5)
plt.tight_layout(pad=0.2)
plt.savefig("figures/paperTF/map_world.pdf", bbox_inches="tight")
plt.savefig("figures/paperTF/map_world.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved map_world")

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig = plt.figure(figsize=nature_figsize(cols=1, height_mm=100))
ax = plt.axes(projection=ccrs.NorthPolarStereo())

ax.set_extent([-180, 180, 30, 90], crs=ccrs.PlateCarree())

# ── base map ──────────────────────────────────────────────────────────────
ax.add_feature(cfeature.OCEAN, facecolor="#daeeff", zorder=0)
ax.add_feature(cfeature.LAND, facecolor="#f0f0f0", zorder=0)
ax.add_feature(cfeature.COASTLINE, linewidth=0.3, zorder=1)
ax.add_feature(cfeature.BORDERS, linewidth=0.2, zorder=1)

# ── glacier centroids ─────────────────────────────────────────────────────
for label, cfg_gl in GLACIERS_TO_PLOT.items():
    gdf_target = cfg_gl["gdf"][cfg_gl["gdf"]["RGIId"] == cfg_gl["rgi"]]
    if gdf_target.empty:
        print(f"Warning: {cfg_gl['rgi']} not found, skipping.")
        continue
    centroid = gdf_target.geometry.centroid.iloc[0]
    ax.plot(
        centroid.x,
        centroid.y,
        marker="*",
        markersize=10,
        color=cfg_gl["color"],
        markeredgecolor="white",
        markeredgewidth=0.3,
        transform=ccrs.PlateCarree(),
        zorder=4,
        label=label,
    )

# ── legend ────────────────────────────────────────────────────────────────
ax.legend(
    loc="lower left",
    fontsize=5,
    frameon=False,
    markerscale=1.0,
    handletextpad=0.3,
)

apply_nature_style(ax, fontsize=5)
plt.tight_layout(pad=0.2)
plt.savefig("figures/paperTF/map_world.pdf", bbox_inches="tight")
plt.savefig("figures/paperTF/map_world.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved map_world")

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

df_stakes = pd.read_csv(
    os.path.join(cfg.dataPath, 'WGMS/raw/mass_balance_point.csv'))

df_RGIId = pd.read_csv(cfg.dataPath + 'WGMS/raw/glacier.csv')

# Create a mapping dictionary from id to rgi60_ids
id_to_rgi_map = dict(zip(df_RGIId['id'], df_RGIId['rgi60_ids']))

# Add the RGIId column to the filtered DataFrame using glacier_id instead of id
df_stakes['RGIId'] = df_stakes['glacier_id'].map(id_to_rgi_map)

# filter to non Nan RGIId
df_stakes = df_stakes[~df_stakes['RGIId'].isna()]
df_stakes.head()

# --- aggregate per glacier (lat/lon + count) ---
df_glacier_counts = (df_stakes.groupby("RGIId").agg(
    lat=("latitude", "mean"),
    lon=("longitude", "mean"),
    n_measurements=("id", "count"),
).reset_index().dropna(subset=["lat", "lon"]))

print(f"Total glaciers with stakes: {len(df_glacier_counts)}")
print(
    f"Total stake measurements:   {df_glacier_counts['n_measurements'].sum():,}"
)

# remove the ax.legend() call entirely from the map section
# and remove the dummy scatter calls too — they're only needed for the legend

# --- plot ---
fig = plt.figure(figsize=nature_figsize(cols=2, height_mm=100))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())

ax.set_global()
ax.add_feature(cfeature.LAND, facecolor="#f0f0f0", linewidth=0)
ax.add_feature(cfeature.OCEAN, facecolor="#d6e8f5", linewidth=0)
ax.add_feature(cfeature.COASTLINE, linewidth=0.3, edgecolor="#999999")
ax.add_feature(cfeature.BORDERS, linewidth=0.2, edgecolor="#cccccc")

sizes = np.clip(df_glacier_counts["n_measurements"], 1, 500)
sizes_scaled = 5 + 80 * (sizes - sizes.min()) / (sizes.max() - sizes.min())

ax.scatter(
    df_glacier_counts["lon"],
    df_glacier_counts["lat"],
    s=sizes_scaled,
    c="gray",
    alpha=0.7,
    linewidths=0.2,
    edgecolors="white",
    transform=ccrs.PlateCarree(),
    zorder=3,
)
# ← no ax.legend() here

plt.tight_layout()
plt.savefig("figures/paperTF/map_wgms_stakes.pdf", bbox_inches="tight")
plt.savefig("figures/paperTF/map_wgms_stakes.png",
            dpi=300,
            bbox_inches="tight")
plt.show()

# ── Standalone legend ─────────────────────────────────────────────────────
fig_leg, ax_leg = plt.subplots(figsize=(60 / 25.4, 10 / 25.4))  # wide, short
ax_leg.axis("off")

for n_val in [10, 100, 500]:
    s_val = 1 + 40 * (n_val - sizes.min()) / (sizes.max() - sizes.min())
    ax_leg.scatter([], [], s=s_val, c="gray", alpha=0.7, label=f"{n_val}")

ax_leg.legend(
    title="Measurements",
    loc="center",
    ncol=3,  # ← all items in one row
    fontsize=NATURE_SPECS["font_min_pt"],
    title_fontsize=NATURE_SPECS["font_min_pt"],
    frameon=True,
    framealpha=0.0,  # ← transparent background
    edgecolor="none",  # ← no border
)

plt.tight_layout(pad=0.1)
plt.savefig("figures/paperTF/map_wgms_legend.pdf", bbox_inches="tight")
plt.savefig("figures/paperTF/map_wgms_legend.png",
            dpi=300,
            bbox_inches="tight")
plt.close()

## TPD schema:

In [ ]:
GLACIERS_TO_PLOT = {
    "Aletsch (CH)": {
        "df":
        df_aletsch,
        "color":
        NATURE_PALETTE["blue"],
        "rgi":
        "RGI60-11.01450",
        "extent": (5.8, 11, 45.5, 48.0),
        "gdf":
        gpd.read_file(
            os.path.join(
                cfg.dataPath,
                "RGI_v6/RGI_11_CentralEurope/11_rgi60_CentralEurope.shp")),
    },
    "Ålfotbreen (NOR)": {
        "df":
        df_gl_nor,
        "color":
        NATURE_PALETTE["vermillion"],
        "rgi":
        "RGI60-08.02992",
        "extent": (4.0, 24.0, 58.0, 71.0),
        "gdf":
        gpd.read_file(
            os.path.join(
                cfg.dataPath,
                "RGI_v6/RGI_08_Scandinavia/08_rgi60_Scandinavia.shp")),
    },
}


In [ ]:
SELECTED_COLS = [
    't2m', 'tp', 'ssrd', 'sshf', 'slhf', 'fal', 'ELEVATION_DIFFERENCE',
    'aspect', 'slope', 'svf'
]
ncols = 3
nrows = int(np.ceil(len(SELECTED_COLS) / ncols))
w, h = nature_figsize(cols=1, height_mm=150)
fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(w * 2, h),  # 178mm wide, 100mm tall
    squeeze=False,
)

for idx, col in enumerate(SELECTED_COLS):
    ax = axes[idx // ncols][idx % ncols]

    all_vals = pd.concat(
        [cfg_gl["df"][col].dropna() for cfg_gl in GLACIERS_TO_PLOT.values()])
    xmin = float(all_vals.min())
    xmax = float(all_vals.max())
    x_grid = np.linspace(xmin, xmax, 500)

    for label, cfg_gl in GLACIERS_TO_PLOT.items():
        vals = cfg_gl["df"][col].dropna().values
        if len(vals) < 10:
            continue
        kde = scipy_stats.gaussian_kde(vals, bw_method=0.3)
        y = kde(x_grid)
        y = y / y.max()
        ax.plot(x_grid, y, color=cfg_gl["color"], linewidth=0.8, label=label)
        ax.fill_between(x_grid, y, alpha=0.08, color=cfg_gl["color"])

    ax.set_ylim(0, 1.15)
    ax.set_xlabel("")

    ax.tick_params(labelsize=8, width=0.4, length=2, direction="in")

    ax.spines[["top", "right", "left", "bottom"]].set_visible(True)
    for spine in ax.spines.values():
        spine.set_linewidth(0.4)
    ax.grid(axis="x", color="#e0e0e0", linewidth=0.3)
    ax.set_axisbelow(True)

    format_axis_ticks(ax, labelsize=6)

for idx in range(len(SELECTED_COLS), nrows * ncols):
    axes[idx // ncols][idx % ncols].axis("off")

handles = [
    plt.Line2D([0], [0], color=COLORS[l], linewidth=1.0, label=l)
    for l in GLACIERS_TO_PLOT
]

plt.tight_layout(h_pad=3.0)

plt.savefig("figures/paperTF/section1_kde_schema_TCD.pdf", bbox_inches="tight")
plt.savefig("figures/paperTF/section1_kde_schema_TCD.png",
            dpi=300,
            bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler


def plot_tsne_three_panels_from_glaciers(
    GLACIERS_TO_PLOT: dict,
    MONTHLY_COLS: list,
    STATIC_COLS: list,
    cfg,
    n_samples_per_glacier: int = 500,
    max_iter: int = 1000,
    perplexity: int = 30,
):
    """
    Three-panel t-SNE (joint, climate-only, topo-only) using the same
    GLACIERS_TO_PLOT dict and colors as the KDE panels in 1_0_Intro_figures.
    One point = one row from the glacier monthly grid dataframes.
    """
    feat_cols_all = MONTHLY_COLS + STATIC_COLS
    feat_cols_clim = MONTHLY_COLS
    feat_cols_topo = STATIC_COLS

    panel_defs = [
        ("Joint\n(climate + topo)", feat_cols_all),
        ("Climate only", feat_cols_clim),
        ("Topographic only", feat_cols_topo),
    ]

    # --- pool subsampled data from each glacier ---
    frames, labels_list, colors_list = [], [], []
    for label, cfg_gl in GLACIERS_TO_PLOT.items():
        df = cfg_gl["df"].dropna(subset=feat_cols_all)
        df_s = df.sample(
            min(n_samples_per_glacier, len(df)),
            random_state=cfg.seed,
        )
        frames.append(df_s)
        labels_list.extend([label] * len(df_s))
        colors_list.extend([cfg_gl["color"]] * len(df_s))

    df_all = pd.concat(frames, ignore_index=True)
    labels_arr = np.array(labels_list)
    colors_arr = np.array(colors_list)

    # --- figure ---
    w, h = nature_figsize(cols=2, height_mm=55)
    fig, axes = plt.subplots(1, 3, figsize=(w, h))

    for ax, (title, feat_cols) in zip(axes, panel_defs):
        X = StandardScaler().fit_transform(df_all[feat_cols].values)

        perp = min(perplexity, len(X) // 4)
        emb = TSNE(
            n_components=2,
            perplexity=perp,
            max_iter=max_iter,
            random_state=cfg.seed,
        ).fit_transform(X)

        # plot each glacier separately so legend entries are clean
        for label, cfg_gl in GLACIERS_TO_PLOT.items():
            mask = labels_arr == label
            ax.scatter(
                emb[mask, 0],
                emb[mask, 1],
                c=cfg_gl["color"],
                label=label,
                alpha=0.4,
                s=6,
                linewidths=0,
                rasterized=True,
            )

        ax.set_title(title, fontsize=NATURE_SPECS["font_max_pt"])
        ax.set_xlabel("t-SNE 1", fontsize=NATURE_SPECS["font_min_pt"])
        ax.set_ylabel("t-SNE 2", fontsize=NATURE_SPECS["font_min_pt"])
        apply_nature_style(ax, fontsize=NATURE_SPECS["font_min_pt"], box=True)

    # shared legend below
    handles = [
        plt.Line2D([0], [0],
                   marker="o",
                   color="w",
                   markerfacecolor=cfg_gl["color"],
                   markersize=4,
                   label=label) for label, cfg_gl in GLACIERS_TO_PLOT.items()
    ]
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=len(GLACIERS_TO_PLOT),
        fontsize=NATURE_SPECS["font_min_pt"],
        frameon=False,
        bbox_to_anchor=(0.5, -0.08),
    )

    plt.tight_layout(rect=[0, 0.05, 1, 1])
    return fig


fig_tsne = plot_tsne_three_panels_from_glaciers(
    GLACIERS_TO_PLOT=GLACIERS_TO_PLOT,
    MONTHLY_COLS=MONTHLY_COLS,
    STATIC_COLS=STATIC_COLS,
    cfg=cfg,
)

plt.savefig("figures/paperTF/section1_tsne_panels.pdf", bbox_inches="tight")
plt.savefig("figures/paperTF/section1_tsne_panels.png",
            dpi=NATURE_SPECS["dpi"],
            bbox_inches="tight")
plt.show()

In [ ]:
GLACIERS_TO_PLOT = {
    "Aletsch (CH)": {
        "df":
        df_aletsch,
        "color":
        NATURE_PALETTE["blue"],
        "rgi":
        "RGI60-11.01450",
        "extent": (5.8, 11, 45.5, 48.0),
        "gdf":
        gpd.read_file(
            os.path.join(
                cfg.dataPath,
                "RGI_v6/RGI_11_CentralEurope/11_rgi60_CentralEurope.shp")),
    },
    "Kahiltna (ALA)": {
        "df":
        df_gl_alaska,
        "color":
        NATURE_PALETTE["bluish_green"],
        "rgi":
        "RGI60-01.00013",
        "extent": (-170, -142, 55.0, 68.0),
        "gdf":
        gpd.read_file(
            os.path.join(cfg.dataPath,
                         "RGI_v6/RGI_01_Alaska/01_rgi60_Alaska.shp")),
    },
}

In [ ]:
SELECTED_COLS = [
    't2m', 'tp', 'ssrd', 'sshf', 'slhf', 'fal', 'ELEVATION_DIFFERENCE',
    'aspect', 'slope', 'svf'
]
ncols = 3
nrows = int(np.ceil(len(SELECTED_COLS) / ncols))
w, h = nature_figsize(cols=1, height_mm=150)
fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(w * 2, h),  # 178mm wide, 100mm tall
    squeeze=False,
)

for idx, col in enumerate(SELECTED_COLS):
    ax = axes[idx // ncols][idx % ncols]

    all_vals = pd.concat(
        [cfg_gl["df"][col].dropna() for cfg_gl in GLACIERS_TO_PLOT.values()])
    xmin = float(all_vals.min())
    xmax = float(all_vals.max())
    x_grid = np.linspace(xmin, xmax, 500)

    for label, cfg_gl in GLACIERS_TO_PLOT.items():
        vals = cfg_gl["df"][col].dropna().values
        if len(vals) < 10:
            continue
        kde = scipy_stats.gaussian_kde(vals, bw_method=0.3)
        y = kde(x_grid)
        y = y / y.max()
        ax.plot(x_grid, y, color=cfg_gl["color"], linewidth=0.8, label=label)
        ax.fill_between(x_grid, y, alpha=0.08, color=cfg_gl["color"])

    ax.set_ylim(0, 1.15)
    ax.set_xlabel("")

    ax.tick_params(labelsize=8, width=0.4, length=2, direction="in")

    ax.spines[["top", "right", "left", "bottom"]].set_visible(True)
    for spine in ax.spines.values():
        spine.set_linewidth(0.4)
    ax.grid(axis="x", color="#e0e0e0", linewidth=0.3)
    ax.set_axisbelow(True)

    format_axis_ticks(ax, labelsize=6)

for idx in range(len(SELECTED_COLS), nrows * ncols):
    axes[idx // ncols][idx % ncols].axis("off")

handles = [
    plt.Line2D([0], [0], color=COLORS[l], linewidth=1.0, label=l)
    for l in GLACIERS_TO_PLOT
]

plt.tight_layout(h_pad=3.0)

plt.savefig("figures/paperTF/section1_kde_schema_TCD.pdf", bbox_inches="tight")
plt.savefig("figures/paperTF/section1_kde_schema_TCD.png",
            dpi=300,
            bbox_inches="tight")
plt.show()